In [3]:
from pathlib import Path
import platform

# If you're on a local Windows kernel, this path should work directly:
LOCAL_VEHICLES_FOLDER = Path(r"C:\Users\E5\Desktop\Ratul\vechicals_works\Vehicles")

# If you're on a Colab-like kernel (cwd like '/content'), put the folder on Drive and set this:
COLAB_VEHICLES_FOLDER = Path("/content/drive/MyDrive/vechicals_works/Vehicles")

# Candidate folder names (some projects use an emoji in the folder name)
FOLDER_NAMES = ["Vehicles", "Vehicles 🚗"]

# Build a list of candidate folders to try
candidates: list[Path] = []

# 1) Explicit paths
candidates.append(LOCAL_VEHICLES_FOLDER)
candidates.append(COLAB_VEHICLES_FOLDER)

# 2) Relative/fallback paths (when running from the project directory)
for name in FOLDER_NAMES:
    candidates.extend([
        Path(name),
        Path.cwd() / name,
        Path.cwd().parent / name,
        Path(r"c:\Users\E5\Desktop\Ratul\vechicals_works") / name,
        Path("/mnt/c/Users/E5/Desktop/Ratul/vechicals_works") / name,
    ])

# 3) If Drive is mounted, also try a few common roots
drive_root = Path("/content/drive/MyDrive")
if drive_root.exists():
    for name in FOLDER_NAMES:
        candidates.extend([
            drive_root / "vechicals_works" / name,
            drive_root / name,
        ])

# De-duplicate while preserving order
seen: set[str] = set()
unique_candidates: list[Path] = []
for p in candidates:
    key = str(p)
    if key not in seen:
        seen.add(key)
        unique_candidates.append(p)

vehicles_folder = next((p for p in unique_candidates if p.exists() and p.is_dir()), None)

if vehicles_folder is None:
    cwd = Path.cwd()
    print("Kernel OS:", platform.platform())
    print("Current working directory:", cwd)
    print("\nTried these Vehicles folder paths:")
    for p in unique_candidates:
        print(" -", p)
    if str(cwd).startswith("/content"):
        print("\nFix options:")
        print("1) In VS Code, switch to a LOCAL Windows Python kernel and rerun.")
        print("2) Or upload/copy your Vehicles folder into this environment.")
        print("   If using Google Drive, run Cell 2 to mount it, then update COLAB_VEHICLES_FOLDER above.")
    raise FileNotFoundError("Could not find an accessible Vehicles folder from the current kernel environment.")

# --- Count images (recursive, includes subfolders) ---
image_extensions = {".jpg", ".jpeg", ".png", ".gif", ".bmp", ".webp", ".tiff", ".tif", ".jfif", ".heic"}
image_files = sorted(
    p
    for p in vehicles_folder.rglob("*")
    if p.is_file() and p.suffix.lower() in image_extensions
)
image_count = len(image_files)

print("Vehicles folder:", vehicles_folder)
print("Total images:", image_count)


Vehicles folder: C:\Users\E5\Desktop\Ratul\vechicals_works\Vehicles
Total images: 500


# Face blur + serial rename (shuffled order)
This step first **shuffles the images**, then checks each image for **human faces**.
- If face(s) are found: it blurs **only the face bounding boxes** (vehicles are not blurred).
- If no face is found: it still saves the image (unchanged).

All outputs are saved into `updated_datasets/` with serial names (`001.jpg`, `002.jpg`, …) and a `mapping.csv` (old path → new name).

In [5]:
# If you get an ImportError for cv2, uncomment the next line and run this cell again:
# %pip install opencv-python

from __future__ import annotations

from pathlib import Path
import csv
import math
import random

try:
    import cv2
except ImportError as e:
    raise ImportError(
        "OpenCV (cv2) is required. Run: %pip install opencv-python then rerun this cell."
    ) from e


# -------------------- Settings --------------------
OUTPUT_FOLDER = vehicles_folder.parent / "updated_datasets"
MAPPING_CSV = OUTPUT_FOLDER / "mapping.csv"

# Save outputs as JPG (keeps filenames consistent like 001.jpg, 002.jpg, ...)
OUTPUT_EXT = ".jpg"
JPG_QUALITY = 98  # 0-100

# Process ALL images by default
MAX_IMAGES: int | None = None

# Shuffle settings (set seed for repeatable shuffle)
SHUFFLE_IMAGES = True
SHUFFLE_SEED = 42

# Clear previous outputs so you don't end up with an old 20-image folder view
CLEAR_OUTPUT_FOLDER = True

# Face detector tuning (Haar cascade)
SCALE_FACTOR = 1.1
MIN_NEIGHBORS = 5
MIN_FACE_SIZE = (30, 30)  # (w, h) in pixels

# Blur strength: higher k => stronger blur. Must be odd numbers.
BLUR_KERNEL = (51, 51)

# Always save images, even when no face is found
SAVE_WHEN_NO_FACE = True


# -------------------- Helpers --------------------
def _ensure_odd_kernel(k: tuple[int, int]) -> tuple[int, int]:
    kx, ky = k
    if kx % 2 == 0:
        kx += 1
    if ky % 2 == 0:
        ky += 1
    return (kx, ky)


def blur_faces_bgr(image_bgr, face_bboxes: list[tuple[int, int, int, int]], blur_kernel: tuple[int, int]):
    """Blur only the face rectangles on a BGR image."""
    blur_kernel = _ensure_odd_kernel(blur_kernel)
    h, w = image_bgr.shape[:2]
    for (x, y, fw, fh) in face_bboxes:
        x0 = max(0, x)
        y0 = max(0, y)
        x1 = min(w, x + fw)
        y1 = min(h, y + fh)
        if x1 <= x0 or y1 <= y0:
            continue
        roi = image_bgr[y0:y1, x0:x1]
        roi_blurred = cv2.GaussianBlur(roi, blur_kernel, 0)
        image_bgr[y0:y1, x0:x1] = roi_blurred
    return image_bgr


# Load face cascade provided by OpenCV
cascade_path = Path(cv2.data.haarcascades) / "haarcascade_frontalface_default.xml"
if not cascade_path.exists():
    raise FileNotFoundError(f"OpenCV haarcascade not found: {cascade_path}")

face_cascade = cv2.CascadeClassifier(str(cascade_path))
if face_cascade.empty():
    raise RuntimeError("Failed to load Haar cascade for face detection.")


# -------------------- Process --------------------
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)

if CLEAR_OUTPUT_FOLDER:
    for old in OUTPUT_FOLDER.glob(f"*{OUTPUT_EXT}"):
        try:
            old.unlink()
        except OSError:
            pass
    if MAPPING_CSV.exists():
        try:
            MAPPING_CSV.unlink()
        except OSError:
            pass

# Start from the full list, then shuffle (if enabled), then slice (if MAX_IMAGES set)
all_images = list(image_files)
if SHUFFLE_IMAGES:
    rng = random.Random(SHUFFLE_SEED)
    rng.shuffle(all_images)

total = len(all_images)
to_process = all_images[: (MAX_IMAGES if MAX_IMAGES is not None else total)]

# Determine serial width (001 vs 000001) based on how many you'll write
width = max(3, int(math.log10(len(to_process))) + 1) if len(to_process) > 0 else 3

processed = 0
skipped_unreadable = 0
faces_total = 0

with MAPPING_CSV.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(
        f,
        fieldnames=["serial", "new_name", "original_path", "faces_detected"],
    )
    writer.writeheader()

    for serial, src_path in enumerate(to_process, start=1):
        img_bgr = cv2.imread(str(src_path))
        if img_bgr is None:
            skipped_unreadable += 1
            continue

        gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(
            gray,
            scaleFactor=SCALE_FACTOR,
            minNeighbors=MIN_NEIGHBORS,
            minSize=MIN_FACE_SIZE,
        )

        face_bboxes = [(int(x), int(y), int(w), int(h)) for (x, y, w, h) in faces]
        if face_bboxes:
            img_bgr = blur_faces_bgr(img_bgr, face_bboxes, BLUR_KERNEL)
            faces_total += len(face_bboxes)

        if (len(face_bboxes) == 0) and (not SAVE_WHEN_NO_FACE):
            continue

        new_name = f"{serial:0{width}d}{OUTPUT_EXT}"
        dst_path = OUTPUT_FOLDER / new_name

        ok = cv2.imwrite(str(dst_path), img_bgr, [int(cv2.IMWRITE_JPEG_QUALITY), int(JPG_QUALITY)])
        if not ok:
            skipped_unreadable += 1
            continue

        writer.writerow({
            "serial": serial,
            "new_name": new_name,
            "original_path": str(src_path),
            "faces_detected": len(face_bboxes),
        })
        processed += 1

saved_files = len(list(OUTPUT_FOLDER.glob(f"*{OUTPUT_EXT}")))

print("Output folder:", OUTPUT_FOLDER)
print("Mapping CSV:", MAPPING_CSV)
print(f"Images available: {total}")
print(f"Images processed: {len(to_process)} (MAX_IMAGES={MAX_IMAGES})")
print(f"Saved images (this run): {processed}")
print(f"Saved files currently in folder: {saved_files}")
print(f"Unreadable/failed: {skipped_unreadable}")
print(f"Total faces blurred: {faces_total}")
print(f"Shuffled: {SHUFFLE_IMAGES} (seed={SHUFFLE_SEED if SHUFFLE_IMAGES else None})")

Output folder: C:\Users\E5\Desktop\Ratul\vechicals_works\updated_datasets
Mapping CSV: C:\Users\E5\Desktop\Ratul\vechicals_works\updated_datasets\mapping.csv
Images available: 500
Images processed: 500 (MAX_IMAGES=None)
Saved images (this run): 500
Saved files currently in folder: 500
Unreadable/failed: 0
Total faces blurred: 1934
Shuffled: True (seed=42)


In [2]:
# Display detailed statistics
print(f"\nDetailed Statistics:")
print(f"Total Images: {image_count}")
print(f"\nImage extensions found:")

extension_counts = {}
for img in image_files:
    ext = img.suffix.lower()
    extension_counts[ext] = extension_counts.get(ext, 0) + 1

for ext, count in sorted(extension_counts.items()):
    print(f"  {ext}: {count}")


Detailed Statistics:
Total Images: 500

Image extensions found:
  .jpg: 500
